    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.
    
    Task 6 (LS4): Implement a program which, (a) given one of the feature models and (b) a value k,
    – creates (and saves) an image-image similarity matrix,
    – performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this image-image
    similarity matrix
    – stores the latent semantics in a properly named output file
    – lists image-weight pairs, ordered in decreasing order of weights

In [19]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [20]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " image latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 5  image latent semantics under  resnet_output  feature space using:  kmeans


In [21]:
from utils.database_utils import exists, store, compressed_retrieve, compressed_store
from feature_models.feature_matrix.image_image_similarity import ImageImageSimilarity

if exists(f'img_img_{FEATURE_SPACE}.pt'):

    image_feature_vectors = compressed_retrieve(f'img_img_{FEATURE_SPACE}.pt')

else:
    print('Image-Image similiarity matrix for ', FEATURE_SPACE, ' does not exist, creating one.')

    image_similarity_generator = ImageImageSimilarity(feature_vectors)
    image_feature_vectors = image_similarity_generator.get_matrix()

    compressed_store(image_feature_vectors, f'img_img_{FEATURE_SPACE}.pt')

print("Image-Image similarity matrix for image-id 0 as example:\n")
print(image_feature_vectors[0])
print("Shape: ", image_feature_vectors[0][1].shape)


Image-Image similarity matrix for image-id 0 as example:

('Faces', array([1.        , 0.22093663, 0.73819107, ..., 0.30177835, 0.20154907,
       0.15875849]))
Shape:  (4339,)


In [22]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer

elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(image_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(image_feature_vectors)

latent_semantics = reducer.reduce_features(image_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[101.70203328  77.15017289 105.63092207  66.66234574  31.32827793]
 [ 88.67532367  62.81872394  98.33940467  78.0945605   49.88766446]
 [106.33545522  83.84898371 111.03716183  74.58566111  23.61169492]
 ...
 [ 88.0119057   65.99304676  93.95369239  43.04651443  66.40308884]
 [101.86167659  80.05494487 103.79654902  44.98128332  88.49123727]
 [ 88.28642043  57.20897881  90.25979815  47.15191953  74.79762437]]
Shape:  (4339, 5)


In [23]:
store(reducer, f'LS4_{FEATURE_SPACE}_{K}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS4_resnet_output_5_kmeans_reducer.pt 



In [24]:
# List image-weight pairs, ordered in decreasing order of weights

# We are to showcase which image contributes more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_weight_tuples = list(zip(feature_vectors.keys(), similarity_matrix))

print("Image - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  3874 , Weight:  118.29992752454945),	(ID:  6668 , Weight:  118.29690828059952),	(ID:  6258 , Weight:  116.97753334593148),	(ID:  6652 , Weight:  116.13301135331312),	(ID:  5008 , Weight:  116.0321814978314),	(ID:  5188 , Weight:  115.91386145211075),	(ID:  812 , Weight:  115.04359119854868),	(ID:  4164 , Weight:  114.84386945272306),	(ID:  5120 , Weight:  114.32804638519968),	(ID:  4582 , Weight:  114.03744054470933),	(ID:  6402 , Weight:  113.87627031645451),	(ID:  508 , Weight:  113.8714573206062),	(ID:  796 , Weight:  113.81342774885447),	(ID:  6650 , Weight:  113.80662707183232),	(ID:  2770 , Weight:  113.78535305136658),	(ID:  3392 , Weight:  113.7193152649642),	(ID:  370 , Weight:  113.71452569005046),	(ID:  8024 , Weight:  113.66689514578911),	(ID:  6468 , Weight:  113.6513284879821),	(ID:  562 , Weight:  113.64605912559763),	(ID:  6430 , Weight:  113.5997952759965),	(I